In [4]:
!pip install pandas

  Using cached pandas-3.0.2-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 6.1 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 14.4 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]


In [5]:
import pandas as pd

df = pd.read_csv("tox21.csv")
df.head()

,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O


In [6]:
target ="SR-HSE"

In [7]:
df = df[["smiles",target]].dropna()

In [11]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 2.3 MB/s  0:00:16m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 2.2 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [rdkit]32m1/2 [rdkit]


In [8]:
from rdkit import Chem

df["mol"] =df["smiles"].apply(lambda x: Chem.MolFromSmiles(x))

[14:58:47] WARNING: not removing hydrogen atom without neighbors
[14:58:47] Explicit valence for atom # 8 Al, 6, is greater than permitted
[14:58:47] Explicit valence for atom # 3 Al, 6, is greater than permitted
[14:58:47] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:58:48] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:58:48] Explicit valence for atom # 9 Al, 6, is greater than permitted
[14:58:48] Explicit valence for atom # 5 Al, 6, is greater than permitted
[14:58:48] Explicit valence for atom # 16 Al, 6, is greater than permitted


In [9]:
df = df[df["mol"].notnull()]

In [14]:
from rdkit import Chem

def safe_mol(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol
    except:
        return None

df["mol"] = df["smiles"].apply(safe_mol)

# Remove invalid ones
df = df[df["mol"].notnull()]

[15:00:37] WARNING: not removing hydrogen atom without neighbors


In [16]:
final_df = pd.concat([desc_df, df[target]], axis=1)
final_df.head()

,MolWt,LogP,NumHDonors,NumHAcceptors,SR-HSE
0,258.324,1.3424,1.0,5.0,0.0
2,288.475,5.0903,1.0,1.0,0.0
4,206.027,-0.9922,5.0,3.0,0.0
5,290.444,4.8172,0.0,4.0,0.0
6,176.624,1.6141,0.0,2.0,0.0


In [17]:
final_df.describe()

,MolWt,LogP,NumHDonors,NumHAcceptors,SR-HSE
count,6460.000000,6460.000000,6460.000000,6460.000000,6460.000000
mean,252.191550,2.261608,1.062539,3.149071,0.057585
std,136.157035,2.081130,1.485270,2.496427,0.232975
min,9.012000,-17.406400,0.000000,0.000000,0.000000
25%,156.269000,1.105400,0.000000,2.000000,0.000000
50%,221.665000,2.254035,1.000000,3.000000,0.000000
75%,317.473750,3.450625,2.000000,4.000000,0.000000
max,1550.188000,15.879200,24.000000,40.000000,1.000000


In [18]:
print(len(df))

6460


In [19]:
desc_df = df["mol"].apply(get_descriptors).apply(pd.Series)